In [1]:
# Auto-generated by build_and_verify.py — do not hand-edit.
# Rebuild with: python build_and_verify.py lesson_01_intro_and_statistics.yaml

from cs470_engine import Worksheet
from cs470_engine.plot_style import apply_default_style

apply_default_style()

ws = Worksheet.load('lesson_01_intro_and_statistics.yaml')
ws.start()


$\def\AA{\text{Å}}$

# Lesson 01: Introduction to Atomic-Scale Simulations, Phase Space & Probability Statistics

_Session f9a3cf72 · Estimated time: ~120 minutes · 40 problems_

**Recording your answers.** Submit each problem as you go. Your answers are saved to the workspace as you submit, *and* the running record at the bottom of the notebook keeps your progress. When you finish, run the last two cells — **`show_record()`** and **`finalize()`** — to record your participation. **If you reopened this workspace** (or the kernel restarted), use **Kernel → Restart Kernel and Run All Cells** before finalizing, so every answer is reloaded.

### Learning Objectives
- Explain why atomic-scale simulations bridge microscopic physical laws (Schrödinger, Boltzmann, Newton's F=ma) and macroscopic materials properties across length and time scales.
- Calculate supercomputing particle-step budgets (N_particles × N_steps) using Moore's Law scaling and method-specific force costs from Ab Initio DFT to Classical MD and KMC.
- Formulate classical phase space S = {r^N, p^N}, distinguish microstates from equilibrium macrostates, and analyze constant-energy trajectories of the harmonic oscillator.
- Compute expectation values, raw and central moments, variance σ² = ⟨X²⟩ − ⟨X⟩², covariance matrices V_ij, correlation coefficients ρ, and variances of linear combinations.
- Derive Gauss's Central Limit Theorem (CLT) via characteristic functions and cumulants (κ_n(y) = κ_n / N^{n-1}) and compute the standard error of the sample mean σ / √N.
- Verify the three analytical conditions required for the CLT and explain why heavy-tailed distributions like the Cauchy/Lorentzian p(x) = 1/[π(1+x²)] violate finite variance and fail to narrow with N.


In [2]:
ws.render_section('sec_intro_multiscale')

## 1. Why Atomic-Scale Simulations? Bridging Length and Time Scales

Welcome to **MSE 485 / CSE 485 / PHYS 466: Atomic Scale Simulations**! Whether we are designing radiation-tolerant nuclear alloys, solid-state battery electrolytes, or defect-free semiconductor interfaces, materials science rests on a fundamental tension nicely summarized by Paul Dirac in 1929:

> *"The underlying physical laws necessary for the mathematical theory of a large part of physics and the whole of chemistry are thus completely known, and the difficulty is only that the exact application of these laws leads to equations much too complicated to be soluble."* — **P. A. M. Dirac** (*Proc. R. Soc. Lond. A*, 1929)

As Frank Wilczek later quipped (*Physics Today*, 2003), we can measure the progress of physics by the problems that *cannot* be solved exactly: in Newtonian gravity, the two-body problem is solvable in closed form while the three-body problem is non-integrable; in general relativity, even two bodies require numerical relativity; and in quantum field theory, the vacuum itself is intractable! For a macroscopic crystal containing $\sim 10^{23}$ interacting nuclei and electrons, analytical closed-form solutions are impossible.

### Two Philosophies of Materials Modeling
To predict how matter behaves, scientists historically relied on two paradigms:
1. **Semi-Empirical Models:** Measure experimental data, invent a phenomenological formula with fitted parameters to mimic the observation, and interpolate. **Fatal flaw:** you *cannot reliably extrapolate* a semi-empirical model outside the thermodynamic or structural regime where its parameters were fit.
2. **First-Principles (*Ab Initio*) & Atomistic Simulations:** Start from the foundational laws of physics — **Schrödinger's equation** (quantum mechanics), **Newton's equations of motion** ($\mathbf{F} = m\mathbf{a}$), **Maxwell's equations** (electromagnetism), and **Boltzmann's statistical mechanics** — and solve the many-body mathematical problem numerically on a computer.

### The Hierarchy of Length and Time Scales
Why do we focus on the **atomic scale**?
- An **atomic nucleus** has a radius of roughly $10\text{ fm} = 10^{-14}\text{ m} = 10^{-5}\text{ Å}$.
- An **atom** (determined by its electronic shells, such as the $K$ and $L$ shells in Bohr's 1913 model) has a radius of **$1\text{ to a few Å}$** ($1\text{ Å} = 10^{-10}\text{ m} = 0.1\text{ nm}$).
- Atomic vibrations (phonons) occur on **femtosecond to picosecond** timescales ($1\text{ fs} = 10^{-15}\text{ s}$, $1\text{ ps} = 10^{-12}\text{ s}$), requiring an integration time step of $\Delta t \sim 10^{-16}\text{ to }10^{-15}\text{ s}$.

Because no single computational method can resolve sub-femtosecond electronic motion across a meter-scale engineering component for hours, computational materials science is organized into a **multiscale hierarchy**:

| Simulation Tier | Typical Length Scale | Typical Time Scale | Core Governing Rule & Scaling |
| :--- | :--- | :--- | :--- |
| **Quantum Mechanics / DFT** | $1\text{ Å} - 5\text{ nm}$ ($10^1 - 10^3$ atoms) | $1\text{ fs} - 100\text{ ps}$ | Kohn–Sham Schrödinger eq.; $\mathcal{O}(N_e^3)$ cost |
| **Classical Molecular Dynamics (MD)** | $1\text{ nm} - 1\text{ }\mu\text{m}$ ($10^3 - 10^9$ atoms) | $1\text{ ps} - 1\text{ }\mu\text{s}$ | Newton's $\mathbf{F}=m\mathbf{a}$ with interatomic potentials; $\mathcal{O}(N)$ with neighbor lists |
| **Kinetic Monte Carlo (KMC)** | $1\text{ nm} - 10\text{ }\mu\text{m}$ | $1\text{ }\mu\text{s} - 10^2\text{ s}$ | Stochastic rare-event Markov jumps $T(S_{n+1}\mid S_n)$ |
| **Mesoscale (Phase-Field / DDD)** | $100\text{ nm} - 1\text{ mm}$ | $1\text{ }\mu\text{s} - 1\text{ hr}$ | Coarse-grained order parameters / dislocation lines |
| **Continuum (Finite Elements)** | $10\text{ }\mu\text{m} - 10\text{ m}$ | $1\text{ ms} - \text{years}$ | Conservation PDEs (elasticity, plasticity, heat) |

### Moore's Law and the Supercomputing Particle–Step Budget
In 1965, **Gordon E. Moore** extrapolated data from 1958–1965 and observed that the number of transistors on an integrated circuit microchip **doubles approximately every two years** (while the inflation-adjusted cost per GFLOPS plummets exponentially). If transistor count (or compute throughput $R_{\text{FLOP}}$) doubles every $T_2 = 2\text{ years}$, then after $\Delta t$ years the performance multiplier is:
$$\frac{R(t_0 + \Delta t)}{R(t_0)} = 2^{\Delta t / 2}$$
However, even on a modern **exascale** or **petascale** supercomputer (Top500 class), computer time is strictly finite. If a computer delivers $R_{\text{FLOP}}$ floating-point operations per second for a wall-clock time $t_{\text{wall}}$, and evaluating the force on 1 particle for 1 time step costs $C_{\text{force}}$ FLOPs, then the simulation is constrained by a strict **particle–step budget**:
$$N_{\text{particles}} \times N_{\text{steps}} \le \frac{R_{\text{FLOP}} \cdot t_{\text{wall}}}{C_{\text{force}}}$$
This fundamental hyperbola explains why we can simulate **few particles for a long time** or **many particles for a short time**, but simulating both large $N_{\text{particles}}$ and long $N_{\text{steps}}$ requires algorithmic breakthroughs and statistical mechanics!


In [3]:
ws.render_section('concept_multiscale')

### Interactive Explorer: The Multiscale Map & Supercomputer Budget Calculator

Select a simulation regime and adjust the supercomputer FLOP rate and wall-clock time to see where each method sits in length–time space and how many total **particle-steps** ($N_{\text{particles}} \times N_{\text{steps}}$) you can afford.

**Key Takeaway:** Moving from *ab initio* DFT ($\sim 10^9\text{ FLOPs/atom-step}$) to Classical MD ($\sim 10^3\text{ FLOPs/atom-step}$) increases the affordable $N_{\text{particles}} \times N_{\text{steps}}$ budget by six orders of magnitude ($10^6\times$), trading explicit electronic wavefunctions for parameterized interatomic potentials.

In [4]:
ws.render_section('q_1')

**Q1 · easy** · 💡

In classical and quantum atomistic modeling, what is the approximate ratio of the radius of a typical **atom** ($r_{\text{atom}} \approx 1\text{ Å}$) to the radius of an **atomic nucleus** ($r_{\text{nuc}} \approx 10\text{ fm}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall from the lecture slides that $10\text{ fm} = 10^{-5}\text{ Å}$.

</div>
</details>

- [ ] **(A)** $r_{\text{atom}} / r_{\text{nuc}} \approx 10^5$ (the atom is roughly $100{,}000$ times larger in radius than its nucleus)
- [ ] **(B)** $r_{\text{atom}} / r_{\text{nuc}} \approx 10^2$ (the atom is roughly $100$ times larger in radius than its nucleus)
- [ ] **(C)** $r_{\text{atom}} / r_{\text{nuc}} \approx 10^9$ (the atom is one billion times larger in radius than its nucleus)
- [ ] **(D)** $r_{\text{atom}} / r_{\text{nuc}} \approx 10$ (the nucleus occupies roughly one-tenth of the atomic radius)

In [5]:
ws.render_section('q_2')

**Q2 · easy** · 💡

What is the primary scientific limitation of a purely **semi-empirical model** (where a mathematical formula is invented and fit directly to experimental data) compared to a first-principles or physics-based atomistic simulation?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider what happens when you ask a fitted curve to predict behavior at pressures, temperatures, or defect configurations not present in the fitting dataset.

</div>
</details>

- [ ] **(A)** Semi-empirical models cannot be reliably extrapolated away from the experimental data regime used to fit them.
- [ ] **(B)** Semi-empirical models always require solving the many-body Schrödinger equation at every time step.
- [ ] **(C)** Semi-empirical models are mathematically guaranteed to violate energy conservation in any time-dependent integration.
- [ ] **(D)** Semi-empirical models cannot be evaluated on parallel supercomputers.

In [6]:
ws.render_section('q_3')

**Q3 · medium** · 💡

Suppose a petascale supercomputer allocation provides a sustained performance of $R_{\text{FLOP}} = 2.5 \times 10^{15}\text{ FLOP/s}$ for a wall-clock duration of $t_{\text{wall}} = 10\text{ hours}$ ($36{,}000\text{ s}$). You are running a Classical MD simulation where evaluating forces and updating positions costs $C_{\text{force}} = 5{,}000\text{ FLOPs}$ per atom per time step. If your simulation needs to run for $N_{\text{steps}} = 9 \times 10^6$ time steps (i.e., $9\text{ ns}$ with a $1\text{ fs}$ step), what is the **maximum number of atoms** $N_{\text{particles}}$ (in millions of atoms) that you can simulate?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First compute total FLOPs $= R_{\text{FLOP}} \times 36{,}000\text{ s}$. Divide by $C_{\text{force}} \times N_{\text{steps}}$ to get $N_{\text{particles}}$, then express the answer in millions ($10^6$).

</div>
</details>

_Numeric answer required._

In [7]:
ws.render_section('q_4')

**Q4 · easy** · 💡

According to **Moore's Law** (Gordon Moore, 1965), transistor counts on microchips (and corresponding compute capability for scalable algorithms) double approximately every $2\text{ years}$. If a simulation code can handle a system of $N_0 = 50{,}000$ atoms for a fixed physical duration on a single chip in the year 2010, how many **millions of atoms** could an identically scaled $\mathcal{O}(N)$ code handle in the same wall-clock time $12\text{ years}$ later (in 2022) assuming strict Moore's Law doubling?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In $\Delta t = 12\text{ years}$, how many 2-year doubling periods occur? Compute $N_0 \times 2^{12/2}$ and convert to millions.

</div>
</details>

_Numeric answer required._

In [8]:
ws.render_section('q_5')

**Q5 · medium** · 💡

Select **all** statements that accurately describe the scope of MSE 485 and the multiscale hierarchy of materials simulations:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the 'You will learn / You will NOT learn' slides and the length-vs-time scale map in Lecture 01.

</div>
</details>

- [ ] **(A)** Classical Molecular Dynamics (MD) and Monte Carlo (MC) are the primary simulation paradigms covered in MSE 485, using packages such as LAMMPS and HOOMD.
- [ ] **(B)** Because atomic vibrations occur on femtosecond timescales, explicit Classical MD integration steps must be on the order of $\Delta t \sim 1\text{ fs}$ ($10^{-15}\text{ s}$).
- [ ] **(C)** In a fixed supercomputing budget, doubling the number of time steps $N_{\text{steps}}$ allows you to simultaneously double the number of particles $N_{\text{particles}}$ at no extra cost.
- [ ] **(D)** Quantum Mechanical Density Functional Theory (DFT) resolves electronic orbitals at the Angstrom scale but is computationally restricted to much smaller length and time scales than Classical MD.

In [9]:
ws.render_section('sec_history_and_phase_space')

### A Brief History of Molecular Simulations
Modern atomistic simulation was born alongside the very first electronic computers in the 1950s:
- **1953 — Metropolis, Rosenbluth, Rosenbluth, Teller & Teller:** Introduced the **Markov Chain Monte Carlo (MCMC)** method on the MANIAC computer at Los Alamos, simulating the equation of state of 2D hard disks.
- **1954 — Fermi, Pasta, Ulam (& Tsingou):** Performed numerical experiments on a 1D chain of anharmonic oscillators to test **ergodicity** — discovering quasi-periodic recurrence rather than immediate thermalization!
- **1958 & 1970 — Alder & Wainwright:** Used event-driven **Molecular Dynamics (MD)** to discover the first-order liquid–solid phase transition in hard spheres (even without attractive forces!) and later discovered hydrodynamic "long-time tails" ($\propto t^{-d/2}$) in velocity autocorrelation functions.
- **1960 — Vineyard:** Simulated radiation damage cascades in crystalline copper using continuous-potential MD.
- **1964 & 1971 — Aneesur Rahman:** Performed the first continuous-potential MD simulation of a real liquid (**liquid argon** via a Lennard-Jones potential in 1964) and later molecular **liquid water** (1971).
- **1967 — Loup Verlet:** Formulated the symplectic **Verlet integration algorithm** and neighbor lists that remain the backbone of MD codes today.
- **1980 — Andersen, Rahman & Parrinello:** Formulated extended-Lagrangian **constant-pressure ($NPT$) MD** allowing the simulation box volume and shape to fluctuate dynamically.
- **1983 — Nosé & Hoover:** Formulated deterministic **constant-temperature ($NVT$) thermostats**.
- **1985 — Car & Parrinello:** Unified electronic Density Functional Theory with classical MD (**Ab Initio MD**).

### Characterizing a System: Phase Space $S$, Microstates & Macrostates
How do we mathematically represent a system inside a computer? We define its **phase space** $S$:
- The **phase space** $S$ is the set of *all possible microscopic states* a system can occupy.
- A single point in phase space is called a **microstate**. It contains *"everything you can know"* about the instantaneous state of the system at that moment:
  - **Classical $N$-particle system in 3D:** A microstate is specified by all $N$ particle position vectors $\mathbf{r}^N = \{\mathbf{q}_1, \dots, \mathbf{q}_N\}$ ($3N$ coordinates) and all $N$ momentum vectors $\mathbf{p}^N = \{\mathbf{p}_1, \dots, \mathbf{p}_N\}$ ($3N$ momenta), forming a **$6N$-dimensional phase space** ($2dN$ dimensions in $d$ spatial dimensions).
  - **Ising Spin Lattice:** A microstate is the discrete tuple of spin orientations $\{\sigma_1, \sigma_2, \dots, \sigma_N\}$ where each $\sigma_i \in \{+1, -1\}$ (yielding $2^N$ total microstates).
- By contrast, an **equilibrium macrostate** is characterized by only a handful of macroscopic thermodynamic **state variables** — for instance, particle number $N$, volume $V$, and total energy $E$ in an isolated ($NVE$ microcanonical) system, or $(N, V, T)$ in a constant-temperature canonical system. Enormously many microstates correspond to the same macrostate!

### Worked Example: The 1D Harmonic Oscillator in Phase Space
Consider a single particle of mass $m$ attached to a linear spring of stiffness $k$ (rest position $x_0 = 0$). Newton's equation of motion $F = ma$ is:
$$m\frac{d^2 z}{dt^2} = -kz \quad \Longrightarrow \quad z(t) = A\cos(\omega t) + B\sin(\omega t), \qquad \omega = \sqrt{\frac{k}{m}}$$
Once the initial position $z(0)$ and initial velocity $v(0)$ are specified, the constants $A = z(0)$ and $B = v(0)/\omega$ are fixed, and the **entire future trajectory is deterministically fixed**. Equivalently, for a Newtonian system where kinetic energy $K$ depends only on momentum $p = mv$ and potential energy $U$ depends only on position $x$, we write the **Hamiltonian** (total energy function):
$$\mathcal{H}(x, p) = K(p) + U(x) = \frac{p^2}{2m} + \frac{1}{2}k(x - x_0)^2 = E$$
Dividing both sides by the constant total energy $E > 0$ reveals the geometry of a **constant-energy trajectory in $(x, p)$ phase space**:
$$\frac{(x - x_0)^2}{(\sqrt{2E/k})^2} + \frac{p^2}{(\sqrt{2mE})^2} = 1$$
This is an **ellipse** centered at $(x_0, 0)$ with position semi-axis $x_{\max} = \sqrt{2E/k}$ (where $p=0$, all energy is potential) and momentum semi-axis $p_{\max} = \sqrt{2mE}$ (where $x=x_0$, all energy is kinetic).

### What is a Simulation? State-to-State Mapping $S_{n+1} = T(S_n)$
Abstractly, every simulation is an iterative rule $T$ that advances the internal state $S_n$ at iteration index $n$ ("time" or "Monte Carlo pseudo-time") to the next state $S_{n+1}$:
$$S_0 \longrightarrow S_1 \longrightarrow S_2 \longrightarrow \cdots \longrightarrow S_n \longrightarrow S_{n+1} = T(S_n)$$
- **Deterministic simulation:** $S_{n+1} = T(S_n)$ is uniquely determined by $S_n$ (e.g., integrating Newton's equations via Verlet MD, or cellular automata like **Conway's Game of Life**, where on a 2D grid a live cell survives iff it has 2 or 3 live neighbors, and a dead cell is born iff it has exactly 3 live neighbors).
- **Stochastic simulation:** $S_{n+1}$ is drawn randomly from a transition probability distribution $T(S_{n+1}\mid S_n)$ (e.g., Metropolis Monte Carlo or Brownian/Langevin dynamics).

Even when the update rule $T$ is deterministic and simple, many-body trajectories are chaotically unpredictable in detail — which is why we analyze **both deterministic MD and stochastic MC outputs using the exact same tools of probability and statistics**!


In [10]:
ws.render_section('concept_phase_space')

### Interactive Explorer: Harmonic Oscillator Phase Space Trajectories

Adjust the total energy $E$, spring stiffness $k$, and particle mass $m$ to see how the constant-energy trajectory $\mathcal{H}(x,p) = E$ reshapes in $(x,p)$ phase space.

**Key Takeaway:** Increasing stiffness $k$ compresses the position semi-axis $x_{\max} = \sqrt{2E/k}$ without changing the momentum intercept $p_{\max} = \sqrt{2mE}$, while increasing energy $E$ scales *both* axes by $\sqrt{E}$ (so the enclosed phase-space area grows linearly with $E$: $\text{Area} = 2\pi E / \omega$).

In [11]:
ws.render_section('q_6')

**Q6 · easy** · 💡

In the history of molecular simulations, which landmark 1953 paper introduced the **Markov Chain Monte Carlo (MCMC)** method by simulating 2D hard disks on the MANIAC computer?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Review the 'Short history of Molecular Simulations' timeline in Lecture 01 (Slide 26).

</div>
</details>

- [ ] **(A)** Metropolis, Rosenbluth, Rosenbluth, Teller, and Teller (1953)
- [ ] **(B)** Alder and Wainwright (1958)
- [ ] **(C)** Rahman (1964)
- [ ] **(D)** Car and Parrinello (1985)

In [12]:
ws.render_section('q_7')

**Q7 · easy** · 💡

How many coordinates are required to specify a single **microstate** in the classical phase space $S$ of $N$ point particles moving in $3$-dimensional space?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Each particle has a 3D position vector $\mathbf{q}_i$ and a 3D momentum vector $\mathbf{p}_i$.

</div>
</details>

- [ ] **(A)** $6N$ coordinates ($3N$ position components and $3N$ momentum components)
- [ ] **(B)** $3N$ coordinates (only the 3D positions of the $N$ particles)
- [ ] **(C)** $3$ state variables (energy $E$, volume $V$, and particle number $N$)
- [ ] **(D)** $2^N$ coordinates

In [13]:
ws.render_section('q_8')

**Q8 · medium** · 💡

A 1D classical harmonic oscillator has mass $m = 2.0\text{ kg}$, spring constant $k = 8.0\text{ N/m}$, and total mechanical energy $E = 16.0\text{ J}$. What is the **maximum momentum** $p_{\max}$ (the positive intercept on the momentum axis of its phase-space ellipse, in $\text{kg}\cdot\text{m/s}$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

At the equilibrium position $x = x_0$, potential energy is zero and $\frac{p_{\max}^2}{2m} = E \implies p_{\max} = \sqrt{2mE}$.

</div>
</details>

_Numeric answer required._

In [14]:
ws.render_section('q_9')

**Q9 · medium** · 💡

For the same 1D harmonic oscillator with $m = 2.0\text{ kg}$, $k = 8.0\text{ N/m}$, and total energy $E = 16.0\text{ J}$, what is the **maximum displacement** $x_{\max} = |x - x_0|_{\max}$ (the positive position intercept of its phase-space ellipse, in meters)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

At the turning points, $p = 0$ and $\frac{1}{2}k x_{\max}^2 = E \implies x_{\max} = \sqrt{2E/k}$.

</div>
</details>

_Numeric answer required._

In [15]:
ws.render_section('q_10')

**Q10 · medium** · 💡

If the total energy of a 1D harmonic oscillator is **quadrupled** from $E_1$ to $E_2 = 4E_1$ while keeping mass $m$ and spring constant $k$ fixed, how do the semi-axes $(x_{\max}, p_{\max})$ and the enclosed area $A = \pi x_{\max} p_{\max}$ of its phase-space ellipse change?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Examine the scaling of $x_{\max} = \sqrt{2E/k}$ and $p_{\max} = \sqrt{2mE}$ with $E$.

</div>
</details>

- [ ] **(A)** Both $x_{\max}$ and $p_{\max}$ double ($2\times$), and the enclosed phase-space area quadruples ($4\times$).
- [ ] **(B)** Both $x_{\max}$ and $p_{\max}$ quadruple ($4\times$), and the enclosed phase-space area increases by $16\times$.
- [ ] **(C)** $x_{\max}$ quadruples while $p_{\max}$ remains constant, so the area quadruples.
- [ ] **(D)** Both $x_{\max}$ and $p_{\max}$ increase by $\sqrt{2}\times$, and the enclosed area doubles ($2\times$).

In [16]:
ws.render_section('q_11')

**Q11 · easy** · 💡

In Lecture 01, **Conway's Game of Life** on a 2D grid is presented as an archetype of a simulation $S_{n+1} = T(S_n)$. Which statement correctly characterizes its update rule $T$ and behavior?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Are there any random numbers in Conway's Game of Life, or is the next generation completely determined by the 8 neighbor states?

</div>
</details>

- [ ] **(A)** The update rule $T$ is strictly deterministic and local (depending only on a cell's state and its 8 neighbors), yet it generates complex, seemingly unpredictable emergent dynamics.
- [ ] **(B)** The update rule $T$ is stochastic, flipping cells with a Boltzmann probability $\exp(-\Delta E/k_B T)$.
- [ ] **(C)** Any live cell with 4 or more live neighbors survives to the next generation.
- [ ] **(D)** Because the rule is deterministic, the exact state $S_n$ at $n = 10^6$ can always be written as a simple linear function of $S_0$ without running the iteration.

In [17]:
ws.render_section('q_12')

**Q12 · medium** · 💡

When characterizing a raw time series (**trace**) of an observable such as potential energy $E(t)$ from a molecular dynamics simulation (Lecture 01, Slide 40), which of the following statements are true? Select **all** that apply.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look at the trace and marginal histogram plot on Slide 40 of Lecture 01.

</div>
</details>

- [ ] **(A)** The initial portion of the trace corresponds to a transient **equilibration (warm-up) time** while the system relaxes from its initial state $S_0$, and must be excluded before computing equilibrium averages.
- [ ] **(B)** Projecting the equilibrated trace values onto the vertical axis forms a **histogram** that estimates the equilibrium probability distribution of the observable.
- [ ] **(C)** Once the system reaches thermal equilibrium, the instantaneous potential energy stops fluctuating and becomes a flat horizontal line.
- [ ] **(D)** By the Law of Large Numbers, the statistical error of the sample mean decreases toward zero as the equilibrated trace length $N \to \infty$.

In [18]:
ws.render_section('sec_prob_moments_covariance')

## 3. Probability Distributions, Moments, Cumulants & Multidimensional Covariance

Because atomistic trajectories fluctuate endlessly across phase space, we extract macroscopic materials properties using **probability theory and statistical mechanics**.

### Discrete vs. Continuous Probability Distributions & Normalization
Whether a random variable $X$ takes discrete values $\{x_k\}$ with probabilities $p_k \ge 0$ or continuous values $x \in (-\infty, \infty)$ with a probability density function (PDF) $F^*(x) \ge 0$, total probability must be normalized to unity. This defines the **zeroth raw moment** $I_0$:
$$\sum_k p_k = 1 \qquad \text{or} \qquad I_0 = \int_{-\infty}^{\infty} dx\, F^*(x) = 1$$
If an unnormalized weight function $w(x) \ge 0$ is given (such as the Boltzmann factor $e^{-\mathcal{H}(S)/k_B T}$), we normalize it by dividing by the **partition function** (normalization integral) $Z = \int w(x)\,dx$:
$$F^*(x) = \frac{w(x)}{\int_{-\infty}^{\infty} w(x')\,dx'}$$

### Expectation Values, Raw Moments $I_n$, Central Moments & Cumulants $\kappa_n$
The **expectation value** (ensemble average) of any function $g(X)$ is:
$$\langle g(X) \rangle = \int_{-\infty}^{\infty} dx\, F^*(x)\, g(x) \qquad \left(\text{or } \sum_k p_k\, g(x_k)\right)$$
The **$n$-th raw moment** is $I_n = \langle X^n \rangle = \int_{-\infty}^{\infty} dx\, F^*(x)\, x^n$. From these, we construct the **mean**, **central moments**, and **cumulants** $\kappa_n$ (generated by the Taylor expansion of the log characteristic function $\ln \langle e^{ikX}\rangle = \sum_{n=1}^\infty \kappa_n \frac{(ik)^n}{n!}$):
1. **First Cumulant / Mean ($\mu$):**
   $$\kappa_1 = \mu = I_1 = \langle X \rangle$$
2. **Second Cumulant / Variance ($\sigma^2$):**
   $$\kappa_2 = \sigma^2 = \text{Var}(X) = \langle (X - \mu)^2 \rangle = \langle X^2 - 2\mu X + \mu^2 \rangle = \langle X^2 \rangle - \langle X \rangle^2 = I_2 - I_1^2$$
   The **standard deviation** $\sigma = \sqrt{\kappa_2} = \sqrt{\text{Var}(X)}$ measures the intrinsic width of the fluctuations of $X$ itself. Notice that shifting a random variable by a constant $a$ and scaling by $b$ gives $\langle a + bX \rangle = a + b\mu$ and $\text{Var}(a + bX) = b^2 \text{Var}(X)$.
3. **Third Cumulant (Unnormalized Skewness $\kappa_3$):**
   $$\kappa_3 = \langle (X - \mu)^3 \rangle$$
   Any distribution that is **symmetric** about its mean ($F^*(\mu + \delta x) = F^*(\mu - \delta x)$) has $\kappa_3 = 0$.
4. **Fourth Cumulant (Unnormalized Excess Kurtosis $\kappa_4$):**
   $$\kappa_4 = \langle (X - \mu)^4 \rangle - 3\kappa_2^2$$
   For a **Gaussian (normal) distribution**, $\langle (X-\mu)^4\rangle = 3\sigma^4$, so **all cumulants of order $n \ge 3$ are identically zero** ($\kappa_3 = \kappa_4 = \cdots = 0$).

### Multidimensional Distributions, Covariance Matrix $V_{ij}$ & Linear Combinations
In simulations we rarely measure just one number; we sample an $m$-dimensional vector of observables $\mathbf{X} = (X_1, X_2, \dots, X_m)^T$ (e.g., potential energy, pressure, and magnetization). The scalar variance generalizes to the symmetric, positive semi-definite $m \times m$ **covariance matrix** $V_{ij}$:
$$V_{ij} = \text{Cov}(X_i, X_j) = \big\langle (X_i - \langle X_i \rangle)(X_j - \langle X_j \rangle) \big\rangle = \langle X_i X_j \rangle - \langle X_i \rangle \langle X_j \rangle$$
- The diagonal entries $V_{ii} = \sigma_i^2 = \text{Var}(X_i)$ are the individual variances.
- The off-diagonal entries $V_{ij}$ ($i \ne j$) measure how $X_i$ and $X_j$ fluctuate together. Normalizing by the individual standard deviations yields the dimensionless **Pearson correlation coefficient** $\rho_{ij} \in [-1, +1]$:
  $$\rho_{ij} = \frac{\text{Cov}(X_i, X_j)}{\sigma_i \sigma_j} \quad \Longleftrightarrow \quad \text{Cov}(X_i, X_j) = \rho_{ij}\sigma_i\sigma_j$$
- Just like a moment-of-inertia tensor in mechanics, diagonalizing $V_{ij}$ (or performing Singular Value Decomposition) rotates $(X_1, X_2)$ into **orthogonal principal axes** whose eigenvalues $\lambda_1, \lambda_2$ are the uncorrelated variances along the major and minor axes of the probability ellipse!
- Finally, what is the variance of a **linear combination** $Z = \sum_{i=1}^m c_i X_i$? Expanding $\langle (Z - \langle Z\rangle)^2 \rangle$ gives the fundamental variance propagation formula:
  $$\text{Var}\!\left(\sum_{i=1}^m c_i X_i\right) = \sum_{i=1}^m c_i^2 \text{Var}(X_i) + 2\sum_{1 \le i < j \le m} c_i c_j \text{Cov}(X_i, X_j)$$
  In particular, for two variables $X$ and $Y$:
  $$\text{Var}(c_1 X + c_2 Y) = c_1^2 \sigma_X^2 + c_2^2 \sigma_Y^2 + 2 c_1 c_2 \rho \sigma_X \sigma_Y$$


In [19]:
ws.render_section('concept_covariance')

### Interactive Explorer: 2D Covariance Matrix, Principal Axes & Linear Combinations

Vary the correlation coefficient $\rho$ and marginal standard deviations $\sigma_X, \sigma_Y$ to see how the covariance matrix $V_{ij}$ tilts the 2D Gaussian contour ellipse (Lecture 02, Slide 6) and alters $\text{Var}(X+Y)$ and $\text{Var}(X-Y)$.

**Key Takeaway:** Positive correlation ($\rho > 0$) stretches fluctuations along the $X+Y$ diagonal, making $\text{Var}(X+Y) > \sigma_X^2 + \sigma_Y^2$ while shrinking $\text{Var}(X-Y)$. Quoting individual 1D error bars on $X_1$ and $X_2$ without their off-diagonal covariance $V_{12}$ misses the strong diagonal coupling!

In [20]:
ws.render_section('q_13')

**Q13 · easy** · 💡

A continuous random variable $X$ on the interval $x \in [0, 2]$ has a probability density function of the form $F^*(x) = C x^2$ for $0 \le x \le 2$ (and $0$ otherwise). What must the normalization constant $C$ be so that the zeroth raw moment satisfies $I_0 = \int_0^2 F^*(x)\,dx = 1$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\int_0^2 x^2\,dx = [x^3/3]_0^2 = 8/3$, then solve $C \cdot (8/3) = 1$.

</div>
</details>

_Numeric answer required._

In [21]:
ws.render_section('q_14')

**Q14 · medium** · 💡

For the normalized probability density $F^*(x) = \frac{3}{8}x^2$ on $x \in [0, 2]$, what is the expectation value (mean) $\mu = I_1 = \langle X \rangle$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Evaluate $\langle X \rangle = \int_0^2 x \cdot \frac{3}{8}x^2\,dx = \frac{3}{8}\int_0^2 x^3\,dx$.

</div>
</details>

_Numeric answer required._

In [22]:
ws.render_section('q_15')

**Q15 · medium** · 💡

In a Monte Carlo simulation of a lattice gas, you measure the particle occupancy $X$ of a small sub-volume. Across equilibrium samples, you find the first raw moment $\langle X \rangle = 4.0$ and the second raw moment $\langle X^2 \rangle = 18.25$. What is the **standard deviation** $\sigma$ of $X$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First compute the variance $\sigma^2 = \langle X^2 \rangle - \langle X \rangle^2$, then take its square root.

</div>
</details>

_Numeric answer required._

In [23]:
ws.render_section('q_16')

**Q16 · easy** · 💡

A discrete defect-hop displacement $X$ (in $\text{Å}$) in a 1D crystal takes three possible values in a single Monte Carlo step: $x = -2\text{ Å}$ with probability $0.25$, $x = 0\text{ Å}$ with probability $0.50$, and $x = +2\text{ Å}$ with probability $0.25$. What is the variance $\sigma_X^2$ (in $\text{Å}^2$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

By symmetry $\langle X \rangle = 0$. Compute $\sigma_X^2 = \langle X^2 \rangle = \sum_k p_k x_k^2$.

</div>
</details>

_Numeric answer required._

In [24]:
ws.render_section('q_17')

**Q17 · easy** · 💡

Suppose an energy observable $E$ (in eV) has mean $\mu_E$ and variance $\sigma_E^2$. If you shift the zero of energy by a constant offset $E_0$ and convert units by multiplying by a constant factor $c$, defining $Y = E_0 + cE$, how do the mean and variance of $Y$ relate to those of $E$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Adding a constant shifts the center of the distribution without changing its spread; scaling by $c$ scales deviations $(Y - \mu_Y) = c(E - \mu_E)$.

</div>
</details>

- [ ] **(A)** $\mu_Y = E_0 + c\mu_E$ and $\text{Var}(Y) = c^2 \sigma_E^2$
- [ ] **(B)** $\mu_Y = E_0 + c\mu_E$ and $\text{Var}(Y) = E_0^2 + c^2 \sigma_E^2$
- [ ] **(C)** $\mu_Y = c\mu_E$ and $\text{Var}(Y) = c \sigma_E^2$
- [ ] **(D)** $\mu_Y = E_0 + c\mu_E$ and $\text{Var}(Y) = c \sigma_E^2$

In [25]:
ws.render_section('q_18')

**Q18 · medium** · 💡

For a 1D classical ideal gas at temperature $T$, the velocity component $v_x$ follows the Gaussian Maxwell–Boltzmann distribution with mean $\mu = \langle v_x \rangle = 0$ and variance $\kappa_2 = \sigma^2 = \langle v_x^2 \rangle = 3.0\text{ (km/s)}^2$. Because the distribution is strictly Gaussian, its fourth cumulant is zero ($\kappa_4 = 0$). What is the fourth central moment $\langle v_x^4 \rangle$ (in $\text{(km/s)}^4$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall from Lecture 02 Slide 5 that $\kappa_4 = \langle (x - \mu)^4 \rangle - 3\kappa_2^2$. Set $\kappa_4 = 0$.

</div>
</details>

_Numeric answer required._

In [26]:
ws.render_section('q_19')

**Q19 · medium** · 💡

In a constant-temperature MD simulation, the total energy is $E = K + U$, where kinetic energy $K$ has standard deviation $\sigma_K = 3.0\text{ eV}$, potential energy $U$ has standard deviation $\sigma_U = 4.0\text{ eV}$, and the two are positively correlated with Pearson correlation coefficient $\rho_{KU} = +0.50$. What is the **variance** $\text{Var}(E) = \text{Var}(K + U)$ (in $\text{eV}^2$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use $\text{Var}(K + U) = \sigma_K^2 + \sigma_U^2 + 2\rho_{KU}\sigma_K\sigma_U$.

</div>
</details>

_Numeric answer required._

In [27]:
ws.render_section('q_20')

**Q20 · medium** · 💡

Now suppose you run an isolated microcanonical ($NVE$) MD simulation where a slight numerical integrator drift causes kinetic energy $K$ ($\sigma_K = 2.0\text{ eV}$) and potential energy $U$ ($\sigma_U = 2.0\text{ eV}$) to be strongly *anticorrelated* with $\rho_{KU} = -0.95$. What is the variance of the total energy $\text{Var}(K + U)$ (in $\text{eV}^2$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Substitute $\sigma_K = 2.0$, $\sigma_U = 2.0$, and $\rho_{KU} = -0.95$ into $\sigma_K^2 + \sigma_U^2 + 2\rho_{KU}\sigma_K\sigma_U$.

</div>
</details>

_Numeric answer required._

In [28]:
ws.render_section('q_21')

**Q21 · hard** · 💡

Two simulation observables $X$ and $Y$ have variances $\sigma_X^2 = 5.0$, $\sigma_Y^2 = 10.0$, and covariance $\text{Cov}(X, Y) = -4.0$. Compute the variance of the linear combination $Z = 2X - 3Y$.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Apply $\text{Var}(c_1 X + c_2 Y) = c_1^2 \sigma_X^2 + c_2^2 \sigma_Y^2 + 2 c_1 c_2 \text{Cov}(X, Y)$ with $c_1 = 2$ and $c_2 = -3$. Watch the signs carefully!

</div>
</details>

_Numeric answer required._

In [29]:
ws.render_section('q_22')

**Q22 · hard** · 💡

In a 2D multidimensional CLT analysis (Lecture 02, Slide 6), two observables $(X_1, X_2)$ have the symmetric covariance matrix:
$$V = \begin{pmatrix} 5.0 & 2.0 \\ 2.0 & 2.0 \end{pmatrix}$$
When $V$ is diagonalized to find the principal axes of the error ellipse, what is the **larger principal variance** (the larger eigenvalue $\lambda_1$ of $V$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Solve $\det(V - \lambda I) = (5 - \lambda)(2 - \lambda) - 2^2 = \lambda^2 - 7\lambda + 6 = 0$.

</div>
</details>

_Numeric answer required._

In [30]:
ws.render_section('q_23')

**Q23 · medium** · 💡

Which of the following correctly describes the relationship between **statistical independence** of two random variables $(X, Y)$ and **zero covariance** $\text{Cov}(X, Y) = 0$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

If $P(X,Y) = P_X(X)P_Y(Y)$, does $\langle XY \rangle = \langle X \rangle \langle Y \rangle$? Conversely, what if $X \sim \mathcal{N}(0,1)$ and $Y = X^2$?

</div>
</details>

- [ ] **(A)** If $X$ and $Y$ are statistically independent, then $\text{Cov}(X,Y) = 0$ always; however, $\text{Cov}(X,Y) = 0$ does not generally imply that $X$ and $Y$ are independent (except when $(X,Y)$ are jointly Gaussian).
- [ ] **(B)** Zero covariance $\text{Cov}(X,Y) = 0$ is mathematically equivalent to statistical independence for all probability distributions.
- [ ] **(C)** Independent random variables have $\text{Cov}(X,Y) = 1$, whereas uncorrelated variables have $\text{Cov}(X,Y) = 0$.
- [ ] **(D)** If $\text{Cov}(X,Y) = 0$, then $\text{Var}(X + Y)$ is always zero.

In [31]:
ws.render_section('sec_clt_and_lorentzian')

## 4. Gauss's Central Limit Theorem (CLT), Cumulant Proof & The Cauchy Counterexample

Why does almost every error bar in computational physics rely on the **normal (Gaussian) distribution**? The answer is **Gauss's Central Limit Theorem (CLT)** (Lecture 02, Slides 5–9).

### Characteristic Function & Cumulant Derivation of the CLT
Suppose we draw $N$ **independent and identically distributed (i.i.d.)** samples $(x_1, x_2, \dots, x_N)$ from an equilibrium distribution $F^*(x)$ with finite mean $\kappa_1 = \mu$ and finite variance $\kappa_2 = \sigma^2$, and compute their **sample mean**:
$$y = \frac{1}{N}\sum_{i=1}^N x_i$$
What is the probability density $P(y)$ of the sample mean $y$? Following Lecture 02 (Slide 5), we take the Fourier transform of $F^*(x)$, known as the **characteristic function** $c_x(k)$, and expand its logarithm in **cumulants** $\kappa_n$:
$$c_x(k) = \langle e^{ikx} \rangle = \int_{-\infty}^{\infty} dx\, F^*(x)\, e^{ikx} = \exp\!\left[\sum_{n=1}^{\infty} \kappa_n \frac{(ik)^n}{n!}\right]$$
Characteristic functions have two magical properties for independent variables:
1. **Sums of independent variables multiply characteristic functions:** $c_{x_1 + x_2}(k) = \langle e^{ik(x_1+x_2)}\rangle = \langle e^{ikx_1}\rangle \langle e^{ikx_2}\rangle = c_{x_1}(k) c_{x_2}(k)$.
2. **Scaling a variable by $\alpha$ scales $k$:** $c_{\alpha x_1}(k) = \langle e^{ik(\alpha x_1)}\rangle = c_{x_1}(\alpha k)$.

Applying both rules to the sample mean $y = \frac{1}{N}x_1 + \cdots + \frac{1}{N}x_N$ of $N$ i.i.d. draws gives:
$$c_y(k) = \big[c_x(k/N)\big]^N = \exp\!\left[N \sum_{n=1}^{\infty} \kappa_n \frac{(ik/N)^n}{n!}\right] = \exp\!\left[\sum_{n=1}^{\infty} \frac{\kappa_n}{N^{n-1}} \frac{(ik)^n}{n!}\right]$$
Comparing term-by-term reveals how the cumulants $\kappa_n(y)$ of the sample mean scale with sample size $N$:
$$\boxed{\kappa_n(y) = \frac{\kappa_n}{N^{n-1}}}$$
Let's examine each order $n$ as $N \to \infty$:
- **$n = 1$ (Mean):** $\kappa_1(y) = \frac{\kappa_1}{N^0} = \mu$ (the mean of the average equals the true mean $\mu$).
- **$n = 2$ (Variance of the Mean):** $\kappa_2(y) = \frac{\kappa_2}{N^1} = \frac{\sigma^2}{N}$. Taking the square root gives the **standard error of the mean (SEM)**:
  $$\sigma_y = \sqrt{\kappa_2(y)} = \sqrt{\frac{\kappa_2}{N}} = \frac{\sigma}{\sqrt{N}}$$
- **$n = 3$ (Skewness of the Mean):** $\kappa_3(y) = \frac{\kappa_3}{N^2}$ (decays as $N^{-2}$, faster than the variance!).
- **$n = 4$ (Kurtosis of the Mean):** $\kappa_4(y) = \frac{\kappa_4}{N^3}$ (decays as $N^{-3}$).

Because all higher cumulants ($n \ge 3$) vanish faster than $\kappa_2(y)$ as $N \to \infty$, the standardized distribution of $y$ retains *only* $\kappa_1(y) = \mu$ and $\kappa_2(y) = \sigma^2/N$, which is uniquely the **Normal (Gaussian) distribution**:
$$P(y) = \sqrt{\frac{N}{2\pi \kappa_2}} \exp\!\left[-\frac{(y - \kappa_1)^2}{2\kappa_2 / N}\right]$$
Even for a flat uniform distribution $R_1$, summing just $N=2$ ($R_2$, triangular), $N=3$ ($R_3$, piecewise quadratic), or $N=12$ ($R_{12}$) draws becomes visually indistinguishable from a Gaussian (Slide 8)!

### The Three Strict Conditions Required for the CLT
The Central Limit Theorem is powerful, but it is **not unconditional**. To apply the i.i.d. CLT, three conditions must hold:
1. **Independent draws** (if samples are correlated in time, as in MD/MC, we must replace $N$ by $N_{\text{eff}} = N/\kappa$, which we will master in Lesson 02).
2. **Identically distributed draws** from a stationary distribution $F^*(x)$.
3. **The first three raw moments ($I_0, I_1, I_2$) must exist and be finite** ($\sigma^2 < \infty$):
   - If $I_0 = \int F^*(x)\,dx$ does not exist $\implies$ not normalizable (not a valid probability distribution).
   - If $I_1 = \int x F^*(x)\,dx$ does not exist $\implies$ no well-defined finite mean.
   - If $I_2 = \int x^2 F^*(x)\,dx$ diverges $\implies$ **infinite variance** ($\sigma^2 = \infty$), and the CLT fails!

As shown on Slide 7 of Lecture 02, for $I_2 = \int_{-\infty}^\infty x^2 F^*(x)\,dx$ to converge, $F^*(x)$ must satisfy two analytical regularity conditions:
- **Fast enough tail decay at infinity:**
  $$\lim_{x \to \pm\infty} x^3 F^*(x) = 0$$
  (i.e., for a power-law tail $F^*(x) \sim |x|^{-\alpha}$, we need $I_2 = \int^\infty x^{2-\alpha}\,dx < \infty \iff \alpha > 3$).
- **Non-singular integrable behavior at all finite points $x_0$:**
  $$\lim_{x \to x_0} (x - x_0) F^*(x) = 0$$

### The Lorentzian / Cauchy Counterexample: Infinite Variance in Action
Consider the **Lorentzian (Cauchy) distribution**:
$$F^*(x) = \frac{1}{\pi(1 + x^2)}, \qquad x \in (-\infty, \infty)$$
- **Normalization ($I_0$):** $\int_{-\infty}^\infty \frac{dx}{\pi(1+x^2)} = \frac{1}{\pi}[\arctan x]_{-\infty}^\infty = 1$ (valid PDF!).
- **Tail decay check:** At large $|x|$, $F^*(x) \sim \frac{1}{\pi x^2}$, so $\lim_{x\to\pm\infty} x^3 F^*(x) = \lim_{x\to\pm\infty} \frac{x}{\pi} = \pm\infty \ne 0$!
- **Second moment ($I_2$):** $I_2 = \int_{-\infty}^\infty \frac{x^2}{\pi(1+x^2)}\,dx = \infty$. **The variance is infinite!**
- **What happens when you average $N$ independent Cauchy draws?** The characteristic function of a standard Cauchy variable is $c_x(k) = e^{-|k|}$. Therefore, the characteristic function of the sample mean $y = \frac{1}{N}\sum_{i=1}^N x_i$ is:
  $$c_y(k) = \big[c_x(k/N)\big]^N = \left(e^{-|k|/N}\right)^N = e^{-|k|} = c_x(k)$$
  **Shocking result:** The average of $N = 10{,}000$ independent Cauchy measurements has the **exact same width and distribution** as a single $N = 1$ measurement! In a simulation trace (Slide 9), an infinite-variance observable betrays itself through violent **outlier spikes** and **heavy power-law tails** that prevent the sample mean from ever settling down. That is why we must establish *analytically* that the variance of our estimator is finite!


In [32]:
ws.render_section('concept_clt_lorentzian')

### Interactive Explorer: Central Limit Theorem Convergence vs. The Cauchy/Lorentzian Trap

Compare how averaging $N$ independent samples narrows the distribution of the sample mean $\bar{X}_N$ as $\sigma/\sqrt{N}$ for Uniform, Exponential, and Bimodal distributions — and observe how the **Cauchy/Lorentzian** distribution completely refuses to narrow as $N$ increases!

**Key Takeaway:** As long as $\sigma^2 < \infty$, even strongly skewed (Exponential) or two-peaked (Bimodal) distributions rapidly morph into a bell-shaped Gaussian whose width shrinks as $N^{-1/2}$. When $\sigma^2 = \infty$ (Cauchy/Lorentzian), averaging $N$ samples provides **zero variance reduction**.

In [33]:
ws.render_section('q_24')

**Q24 · medium** · 💡

In the cumulant proof of Gauss's Central Limit Theorem (Lecture 02, Slide 5), if an individual draw $x$ has cumulants $\kappa_n$, how does the **$n$-th cumulant** $\kappa_n(y)$ of the sample mean $y = \frac{1}{N}\sum_{i=1}^N x_i$ of $N$ independent draws scale with $N$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall $c_y(k) = [c_x(k/N)]^N = \exp\left[N \sum_{n=1}^\infty \kappa_n \frac{(ik/N)^n}{n!}\right]$.

</div>
</details>

- [ ] **(A)** $\kappa_n(y) = \dfrac{\kappa_n}{N^{n-1}}$ (so $\kappa_1(y)=\mu$, $\kappa_2(y)=\sigma^2/N$, $\kappa_3(y)=\kappa_3/N^2$, etc.)
- [ ] **(B)** $\kappa_n(y) = \dfrac{\kappa_n}{N^n}$ (so the mean itself shrinks as $\mu/N$)
- [ ] **(C)** $\kappa_n(y) = \dfrac{\kappa_n}{\sqrt{N}}$ for all orders $n \ge 1$
- [ ] **(D)** $\kappa_n(y) = N\kappa_n$ for all orders $n \ge 1$

In [34]:
ws.render_section('q_25')

**Q25 · easy** · 💡

You measure the equilibrium pressure in a Monte Carlo simulation using $N_1 = 250$ independent samples and obtain a standard error of the mean $\sigma_{\bar{P}} = 0.40\text{ GPa}$. By what factor must you multiply the number of independent samples $N$ to reduce the standard error of the mean to $\sigma_{\bar{P}} = 0.04\text{ GPa}$ (a $10\times$ reduction in error bar)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since $\sigma_{\bar{P}} = \sigma_P / \sqrt{N} \propto N^{-1/2}$, reducing $\sigma_{\bar{P}}$ by a factor of $10$ requires increasing $N$ by $10^2$.

</div>
</details>

_Numeric answer required._

In [35]:
ws.render_section('q_26')

**Q26 · medium** · 💡

Suppose a single draw $x$ from a skewed energy distribution has third cumulant $\kappa_3 = 18.0\text{ eV}^3$. If you form the sample mean $y = \frac{1}{N}\sum_{i=1}^N x_i$ over $N = 6$ independent draws, what is the third cumulant $\kappa_3(y)$ of the sample mean $y$ (in $\text{eV}^3$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Use the cumulant scaling rule $\kappa_3(y) = \kappa_3 / N^{3-1} = \kappa_3 / N^2$.

</div>
</details>

_Numeric answer required._

In [36]:
ws.render_section('q_27')

**Q27 · medium** · 💡

In classical random number generation (Lecture 02, Slide 8), summing $N = 12$ independent uniform random numbers $U_i \sim \text{Uniform}(0, 1)$ was historically used as a fast approximation to a Gaussian distribution ($R_{12}$). Given that a single $\text{Uniform}(0,1)$ draw has variance $\sigma_1^2 = \frac{1}{12}$, what is the **standard error** $\sigma_{\bar{U}}$ of the sample mean $\bar{U} = \frac{1}{12}\sum_{i=1}^{12} U_i$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $\sigma_{\bar{U}} = \sqrt{\sigma_1^2 / N} = \sqrt{(1/12) / 12}$.

</div>
</details>

_Numeric answer required._

In [37]:
ws.render_section('q_28')

**Q28 · medium** · 💡

According to the CLT regularity conditions on Lecture 02 Slide 7, what happens if the raw moments $I_n = \int_{-\infty}^{\infty} dx\, F^*(x) x^n$ fail to exist at order $n = 0, 1,$ or $2$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Match $I_0$, $I_1$, and $I_2$ to normalization, mean, and variance.

</div>
</details>

- [ ] **(A)** If $I_0$ is undefined, $F^*(x)$ is not normalizable; if $I_1$ does not exist, there is no finite mean; if $I_2$ does not exist, the variance is infinite and the standard CLT fails.
- [ ] **(B)** If $I_2$ diverges, the sample mean still converges to a Gaussian with standard error $1/\sqrt{N}$, provided $I_0 = 1$.
- [ ] **(C)** Only $I_0$ and $I_1$ need to exist for the standard error $\sigma/\sqrt{N}$ to be finite; $I_2$ only affects skewness.
- [ ] **(D)** We can always determine whether $I_2$ is finite purely by computing the empirical sample variance of a finite simulation trace, without analytical inspection.

In [38]:
ws.render_section('q_29')

**Q29 · hard** · 💡

Suppose a probability density function has power-law tails $F^*(x) \propto |x|^{-\alpha}$ as $x \to \pm\infty$. For what range of the tail exponent $\alpha$ is the distribution **normalizable with a finite mean**, yet has **infinite variance** ($I_0 < \infty,\ |I_1| < \infty$, but $I_2 = \infty$)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In 1D, $\int_1^\infty x^n x^{-\alpha}\,dx < \infty$ requires $n - \alpha < -1$, i.e. $\alpha > n + 1$. Check $n=1$ and $n=2$.

</div>
</details>

- [ ] **(A)** $2 < \alpha \le 3$
- [ ] **(B)** $\alpha > 3$
- [ ] **(C)** $1 < \alpha \le 2$
- [ ] **(D)** $0 < \alpha \le 1$

In [39]:
ws.render_section('q_30')

**Q30 · medium** · 💡

Consider the standard **Cauchy (Lorentzian) distribution** $F^*(x) = \frac{1}{\pi(1 + x^2)}$. If you draw $N = 100$ independent samples $x_1, \dots, x_{100}$ from $F^*(x)$ and compute their average $y = \frac{1}{100}\sum_{i=1}^{100} x_i$, how does the width (e.g., interquartile range or half-width at half-maximum) of the distribution $P(y)$ compare to that of a single draw $x_1$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Check the Interactive Explorer with `Cauchy / Lorentzian` selected, or recall $c_y(k) = (e^{-|k|/N})^N = e^{-|k|}$.

</div>
</details>

- [ ] **(A)** The width of $P(y)$ is identical to the width of a single draw ($1\times$): averaging 100 Cauchy draws produces zero narrowing.
- [ ] **(B)** The width of $P(y)$ is $10\times$ narrower ($1/\sqrt{100}$) than a single draw, just like a Gaussian.
- [ ] **(C)** The width of $P(y)$ is $100\times$ narrower ($1/N$) than a single draw.
- [ ] **(D)** The distribution $P(y)$ becomes a delta function at $y = 0$.

In [40]:
ws.render_section('q_31')

**Q31 · medium** · 💡

When inspecting a raw simulation time series (trace) and its marginal histogram (Lecture 02, Slide 9: *'What does infinite variance look like?'*), what are the two telltale visual signatures that your estimator suffers from **infinite variance**? Select **all** that apply.

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Look directly at the two text annotations on Slide 9 of Lecture 02.

</div>
</details>

- [ ] **(A)** Large, isolated outlier **spikes** in the time series that do not disappear as the simulation runs longer.
- [ ] **(B)** **Long (heavy) tails** on the marginal histogram distribution.
- [ ] **(C)** A perfectly smooth sinusoidal oscillation with constant amplitude.
- [ ] **(D)** A histogram that is strictly bounded between two sharp vertical walls with zero outliers.

In [41]:
ws.render_section('q_32')

**Q32 · hard** · 💡

Suppose a probability density function $F^*(x)$ on $x \in (0, 1]$ behaves near the origin $x_0 = 0$ as $F^*(x) \propto x^{-\beta}$ (with $0 < \beta < 1$). Why does Lecture 02 Slide 7 include the non-singular condition $\lim_{x \to x_0} (x - x_0) F^*(x) = 0$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Consider what happens if $F^*(x)$ has a pole at $x_0$ as strong as $(x - x_0)^{-1}$.

</div>
</details>

- [ ] **(A)** Because if $F^*(x)$ diverges as $(x - x_0)^{-1}$ or faster at an interior point $x_0$, the integral $\int F^*(x)\,dx$ diverges logarithmically (or worse) at $x_0$, destroying normalizability.
- [ ] **(B)** Because any probability density function must be strictly equal to zero at $x = 0$.
- [ ] **(C)** Because a finite discontinuity (step function) at $x_0$ makes the variance infinite.
- [ ] **(D)** Because the characteristic function cannot be defined for functions on bounded intervals.

In [42]:
ws.render_section('sec_confidence_and_synthesis')

## 5. Interpreting Gaussian Error Bars, Confidence Intervals & Synthesis

Let's bring together the statistical mechanics and probability foundations from Lesson 01 and see how we interpret the error bars reported in every simulation paper and homework report.

### What Does a $1\sigma$ Error Bar Actually Mean?
When a simulation reports an equilibrium property as:
$$\langle A \rangle = \bar{A} \pm \sigma_{\bar{A}}$$
where $\bar{A} = \frac{1}{N}\sum_{i=1}^N A_i$ is the sample mean and $\sigma_{\bar{A}} = \frac{\sigma_A}{\sqrt{N_{\text{eff}}}}$ is the **standard error of the mean (SEM)**, Gauss's Central Limit Theorem tells us that for sufficiently large $N_{\text{eff}}$, the estimator $\bar{A}$ is distributed around the true ensemble expectation $\langle A \rangle$ according to a Gaussian distribution with standard deviation $\sigma_{\bar{A}}$:
- **Within $\pm 1\sigma_{\bar{A}}$:** Approximately **$68.3\%$** ($\approx 2/3$) of the time, the true exact answer $\langle A \rangle$ lies within $\pm 1\sigma_{\bar{A}}$ of $\bar{A}$ (and in $\sim 1/3$ of independent runs, the true value lies *outside* the $1\sigma$ error bar!).
- **Within $\pm 2\sigma_{\bar{A}}$:** Approximately **$95.4\%$** of the time, the true answer lies within $\pm 2\sigma_{\bar{A}}$ (so a $2\sigma$ discrepancy occurs naturally by pure statistical chance about **$1$ time in $20$** — or as Prof. Trinkle notes on Slide 17 of Lecture 02, when comparing two noisy datasets $A$ and $B$, a $\sim 2$-error-bar difference happens in roughly $1$ out of $10$ comparisons!).
- **Within $\pm 3\sigma_{\bar{A}}$:** Approximately **$99.7\%$** of the time, the true answer lies within $\pm 3\sigma_{\bar{A}}$.

### Comparing Two Independent Simulation Runs
Suppose Run A estimates $\bar{A} \pm \sigma_{\bar{A}}$ and an independent Run B estimates $\bar{B} \pm \sigma_{\bar{B}}$. To test whether the difference $\Delta = \bar{A} - \bar{B}$ is statistically consistent with zero, we use our linear combination variance formula ($\text{Cov}(\bar{A}, \bar{B}) = 0$ for independent runs):
$$\sigma_{\Delta} = \sqrt{\text{Var}(\bar{A} - \bar{B})} = \sqrt{\sigma_{\bar{A}}^2 + \sigma_{\bar{B}}^2}$$
and compute the standardized $z$-score $z = \frac{|\bar{A} - \bar{B}|}{\sqrt{\sigma_{\bar{A}}^2 + \sigma_{\bar{B}}^2}}$.

> [!IMPORTANT]
> **Distinguishing $\sigma_A$ (Intrinsic Fluctuation Width) vs. $\sigma_{\bar{A}}$ (Error of the Mean):**
> - As you run a simulation $100\times$ longer ($N \to 100N$), the **histogram width** $\sigma_A = \sqrt{\langle A^2\rangle - \langle A\rangle^2}$ of the observable **does NOT shrink to zero** — it converges to a constant physical property of the system (which in canonical ensembles is directly related to response functions like heat capacity $C_V = \sigma_E^2 / (k_B T^2)$!).
> - What shrinks to zero as $N^{-1/2}$ is the **standard error of the mean** $\sigma_{\bar{A}} = \sigma_A / \sqrt{N_{\text{eff}}}$, which measures our uncertainty in the *center* of that histogram.


In [43]:
ws.render_section('q_33')

**Q33 · easy** · 💡

As you increase the number of independent equilibrium samples in a simulation from $N = 1{,}000$ to $N = 1{,}000{,}000$ (keeping temperature $T$, volume $V$, and particle number fixed), what happens to the **sample standard deviation** $s_E$ of the energy histogram versus the **standard error of the mean** $\sigma_{\bar{E}}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Does the width of the energy histogram shrink as you take more samples, or does it approach the true physical width $\sigma_E$?

</div>
</details>

- [ ] **(A)** The histogram standard deviation $s_E$ approaches a constant physical value $\sigma_E$, whereas the standard error of the mean $\sigma_{\bar{E}} = s_E/\sqrt{N}$ shrinks by a factor of $\sqrt{1000} \approx 31.6$.
- [ ] **(B)** Both $s_E$ and $\sigma_{\bar{E}}$ shrink toward zero as $1/\sqrt{N}$.
- [ ] **(C)** The histogram standard deviation $s_E$ shrinks as $1/\sqrt{N}$, whereas $\sigma_{\bar{E}}$ remains constant.
- [ ] **(D)** Both $s_E$ and $\sigma_{\bar{E}}$ grow proportional to $\sqrt{N}$.

In [44]:
ws.render_section('q_34')

**Q34 · easy** · 💡

According to the Central Limit Theorem (Lecture 02, Slide 15), if you repeat an unbiased simulation many times and quote a $\pm 1\sigma_{\bar{A}}$ standard error bar on the sample average $\bar{A}$, approximately what fraction of the time will the true ensemble expectation $\langle A \rangle$ fall **within** $\bar{A} \pm 1\sigma_{\bar{A}}$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Recall the standard normal integral between $-1\sigma$ and $+1\sigma$ (quoted as '~2/3' on Slide 15).

</div>
</details>

- [ ] **(A)** Approximately $68\%$ ($\approx 2/3$) of the time
- [ ] **(B)** Approximately $95\%$ of the time
- [ ] **(C)** $100\%$ of the time, unless there is a bug in the code
- [ ] **(D)** Approximately $50\%$ of the time

In [45]:
ws.render_section('q_35')

**Q35 · medium** · 💡

Two students independently run Monte Carlo simulations of the same 2D Ising lattice at the same temperature using different random number seeds. Student 1 obtains a mean energy per spin of $\bar{E}_1 = -1.420 \pm 0.006$, and Student 2 obtains $\bar{E}_2 = -1.405 \pm 0.008$. What is the combined standard error $\sigma_{\Delta} = \sqrt{\sigma_1^2 + \sigma_2^2}$ of the difference $\Delta = \bar{E}_1 - \bar{E}_2$?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since the two runs are independent, $\text{Var}(\bar{E}_1 - \bar{E}_2) = \sigma_1^2 + \sigma_2^2$. Spot the 3-4-5 right triangle!

</div>
</details>

_Numeric answer required._

In [46]:
ws.render_section('q_36')

**Q36 · medium** · 💡

In the previous problem ($\bar{E}_1 = -1.420 \pm 0.006$ and $\bar{E}_2 = -1.405 \pm 0.008$, with combined standard error $\sigma_\Delta = 0.010$), how many combined standard errors ($z = |\bar{E}_1 - \bar{E}_2| / \sigma_\Delta$) separate the two students' results?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Compute $|{-1.420} - (-1.405)| / 0.010$.

</div>
</details>

_Numeric answer required._

In [47]:
ws.render_section('q_37')

**Q37 · medium** · 💡

On Slide 17 of Lecture 02 (*'Statistical thinking is slippery: be careful'*), Prof. Trinkle warns: *'If things agree too well, something is wrong!'* Suppose a student runs 20 supposedly independent Monte Carlo runs of a noisy liquid and finds that all 20 sample means agree with the literature reference value to within $0.05\,\sigma_{\bar{A}}$. What is the most likely diagnosis?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

In 20 truly independent Gaussian draws, about $1/3$ should fall outside $\pm 1\sigma_{\bar{A}}$, and only $\sim 4\%$ of draws fall within $\pm 0.05\sigma_{\bar{A}}$.

</div>
</details>

- [ ] **(A)** Either the runs are not actually independent (e.g., reusing the same random seed) or the reported error bars $\sigma_{\bar{A}}$ have been grossly overestimated.
- [ ] **(B)** The Central Limit Theorem guarantees that all runs lie within $\pm 0.05\sigma_{\bar{A}}$ once $N > 100$.
- [ ] **(C)** The observable must follow a Cauchy/Lorentzian distribution.
- [ ] **(D)** The simulation time step $\Delta t$ is too large.

In [48]:
ws.render_section('q_38')

**Q38 · hard** · 💡

In a canonical ($NVT$) simulation at temperature $T$, the equilibrium distribution of states $S$ is the Boltzmann distribution $F^*(S) = Z^{-1} \exp(-\mathcal{H}(S)/k_B T)$. Consider a simplified two-level defect system where State 0 has energy $E_0 = 0\text{ eV}$ and State 1 has energy $E_1 = 1.0\text{ eV}$, at a thermal energy of $k_B T = 1.0\text{ eV}$ (so $p_0 = \frac{1}{1 + e^{-1}} \approx 0.73106$ and $p_1 = \frac{e^{-1}}{1 + e^{-1}} \approx 0.26894$). What is the **variance** $\sigma_E^2 = \langle E^2 \rangle - \langle E \rangle^2$ of the energy (in $\text{eV}^2$, to 3 decimal places)?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Since $E \in \{0, 1\}$ is a Bernoulli random variable with $E^2 = E$, we have $\langle E \rangle = p_1$ and $\sigma_E^2 = p_1(1 - p_1) = p_0 p_1$.

</div>
</details>

_Numeric answer required._

In [49]:
ws.render_section('q_39')

**Q39 · hard** · 💡

In an MD simulation of atomic diffusion in a crystal at high temperature, you track the displacement $\Delta x_i = x_i(t) - x_i(0)$ of $N = 400$ independent tracer atoms over a fixed time interval $t$. According to Einstein's diffusion relation (Lecture 02, Slide 3), in $d = 1$ dimension $\langle (\Delta x)^2 \rangle = 2Dt$, with $\langle \Delta x \rangle = 0$. If the diffusion constant is $D = 0.025\text{ Å}^2/\text{ps}$ and $t = 50.0\text{ ps}$, what is the **standard error** $\sigma_{\overline{\Delta x}}$ (in $\text{Å}$) of the sample mean displacement $\overline{\Delta x} = \frac{1}{N}\sum_{i=1}^N \Delta x_i$ across the $N = 400$ atoms?

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

First find the single-atom variance $\sigma_{\Delta x}^2 = \langle (\Delta x)^2 \rangle = 2Dt$. Then compute the standard error of the 400-atom mean: $\sigma_{\overline{\Delta x}} = \sqrt{\sigma_{\Delta x}^2 / N}$.

</div>
</details>

_Numeric answer required._

In [50]:
ws.render_section('q_40')

**Q40 · hard** · 💡

Reviewing the complete workflow of Lesson 01 — from multiscale physical laws and phase space to probability moments and the Central Limit Theorem — select **all** statements that are mathematically and physically rigorous:

<details style="margin: 0.5em 0;">
<summary style="cursor: pointer; user-select: none; display: inline-block; padding: 0.3em 0.7em; margin: 0.2em 0; background: #EEF0F7; border-radius: 4px; font-weight: 500;">💡 Show hint</summary>

<div style="padding: 0.5em 1em;">

Verify each statement against our derivations of covariance, cumulants, and phase space.

</div>
</details>

- [ ] **(A)** For any two random variables $X$ and $Y$ with finite variance, $\text{Var}(X - Y) = \text{Var}(X) + \text{Var}(Y) - 2\text{Cov}(X, Y)$.
- [ ] **(B)** In the Central Limit Theorem for $N$ i.i.d. samples with finite variance, the skewness cumulant $\kappa_3(y)$ of the sample mean decays as $N^{-2}$, which is faster than the variance $\kappa_2(y) \propto N^{-1}$.
- [ ] **(C)** In a classical Hamiltonian system $\mathcal{H}(\mathbf{r}^N, \mathbf{p}^N) = K(\mathbf{p}^N) + U(\mathbf{r}^N)$, each point in the $6N$-dimensional phase space uniquely specifies the instantaneous microstate of the system.
- [ ] **(D)** Averaging $N$ independent samples from any normalized probability distribution $F^*(x)$ is guaranteed to reduce the width of the sample mean distribution by $1/\sqrt{N}$, even if $I_2 = \int x^2 F^*(x)\,dx = \infty$.

In [ ]:
ws.show_record()

In [ ]:
ws.finalize()